# Notebook 1 — Physics Engine Testing

Tests the **Micro-Loop** pipeline (no real video required — uses synthetic data):

| # | Module | What it tests |
|---|--------|---------------|
| 1 |  | Savitzky-Golay velocity/acceleration from synthetic BEV coords |
| 2 |  | Insert state vectors + trajectory window query |
| 3 |  | SoM overlay on a synthetic frame |
| 4 |  | Homography mapping with a temp calibration file |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
print("sys.path set — imports ready")

## 1. KinematicEstimator

Feed 35 frames of a vehicle with uniform acceleration (a=2 m/s²). After the 15-frame window fills, velocity and acceleration should be non-zero.

In [ ]:
from src.physics_engine.kinematics import KinematicEstimator

FPS = 30.0
DT  = 1.0 / FPS
A_TRUE = 2.0  # m/s^2

estimator = KinematicEstimator(fps=FPS, window_length=15, polyorder=3)

results = []
for i in range(35):
    t = i * DT
    x = 0.5 * A_TRUE * t**2  # s = 1/2 * a * t^2
    state = estimator.update({4: (x, 10.0)})
    results.append(state.get(4))

x, y, vx, vy, ax, ay = results[-1]
speed = (vx**2 + vy**2)**0.5
print(f"Final: pos=({x:.2f}, {y:.2f})  vx={vx:.3f} m/s  ax={ax:.3f} m/s^2")
print(f"Expected speed ~{A_TRUE * (35*DT):.3f} m/s")

assert speed > 0.5, f"Speed too low: {speed}"
assert abs(ax) > 0.1, f"Acceleration too low: {ax}"
print("PASS: KinematicEstimator produces correct velocity & acceleration")

In [ ]:
# Stale trajectory cleanup
estimator2 = KinematicEstimator(fps=FPS)
for i in range(5):
    estimator2.update({99: (float(i), 0.0)})

assert 99 in estimator2.trajectories
estimator2.update({})  # vehicle left frame
assert 99 not in estimator2.trajectories
print("PASS: Stale trajectory for track 99 cleaned up correctly")

## 2. DuckDBClient — Insert & Query

Insert 50 frames for 2 vehicles then query a trajectory window (simulates the LangGraph  tool).

In [ ]:
import duckdb
from src.memory_layer.duckdb_client import DuckDBClient

TEST_DB = "/tmp/test_physics.duckdb"
if os.path.exists(TEST_DB): os.remove(TEST_DB)

db  = DuckDBClient(db_path=TEST_DB)
kin = KinematicEstimator(fps=FPS)

for frame_id in range(50):
    ts = frame_id * DT
    coords = {
        4: (0.5 * 2.0 * ts**2,  5.0),
        9: (50.0 - 0.5 * 1.5 * ts**2, 8.0),
    }
    sv = kin.update(coords)
    db.insert_state_vectors(ts, frame_id, sv)

db.close()
print("Inserted 50 frames for Vehicles 4 and 9")

In [ ]:
# Verify rows and query a window
conn = duckdb.connect(TEST_DB)
count = conn.execute("SELECT COUNT(*) FROM vehicle_trajectories").fetchone()[0]
print(f"Total rows: {count}")

df4 = conn.execute("SELECT timestamp, pos_x, vel_x FROM vehicle_trajectories WHERE track_id=4 ORDER BY timestamp").df()
print(f"Vehicle 4 — {len(df4)} rows, max vel_x={df4.vel_x.max():.3f} m/s")
conn.close()

# Trajectory window (same API as the agent tool)
db2 = DuckDBClient(db_path=TEST_DB)
window = db2.get_trajectory_window(0.2, 0.8, track_id=4)
print(f"Trajectory window 0.2-0.8s for Vehicle 4: {len(window)} rows")
print(window[["timestamp","pos_x","vel_x","accel_x"]].head())

assert count > 0 and len(window) > 0
print("PASS: DuckDB insert + trajectory window query correct")
db2.close()

## 3. AdaptiveRenderer — SoM overlays on a synthetic frame

In [ ]:
import cv2
import matplotlib.pyplot as plt
from src.semantic_abstractor.set_of_mark import AdaptiveRenderer, RenderContext

# Dark-gray 720p synthetic road frame
frame    = np.full((720, 1280, 3), fill_value=50, dtype=np.uint8)
original = frame.copy()

# Synthetic tracker boxes: [x1, y1, x2, y2, track_id]
fake_tracks = np.array([
    [100, 200, 250, 320, 4],
    [500, 300, 650, 420, 9],
    [900, 100, 1000, 200, 3],
])

renderer = AdaptiveRenderer()
ctx      = RenderContext()
ctx.update(fake_tracks, timestamp=10.0)
renderer.render(frame, ctx)

pixel_diff = np.sum(np.abs(frame.astype(int) - original.astype(int)))
print(f"Pixel diff after render: {pixel_diff}")
assert pixel_diff > 0, "Renderer made no changes"

print(f"Tracks drawn: {ctx.density} | Timestamp: {ctx.timestamp}s")
print("PASS: AdaptiveRenderer modifies frame correctly")

plt.figure(figsize=(12, 4))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title("Synthetic SoM frame — IDs 3, 4, 9")
plt.axis("off")
plt.tight_layout()
plt.show()

## 4. CoordinateTransformer — Homography with a mock calibration

Write a temp , load it, and verify pixel → real-world mapping.

In [ ]:
import yaml, tempfile
from src.physics_engine.homography import CoordinateTransformer

# 4-point pixel <-> real-world correspondence for a 1280x720 image
mock_calib = {
    "pixel_points": [
        [0, 720], [1280, 720], [1280, 0], [0, 0]
    ],
    "world_points": [
        [0.0, 0.0], [30.0, 0.0], [30.0, 20.0], [0.0, 20.0]
    ]
}

with tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False) as f:
    yaml.dump(mock_calib, f)
    calib_path = f.name

transformer = CoordinateTransformer(calib_path)

# 3 vehicles at various pixel positions
fake_boxes = np.array([
    [50,  680, 150, 720, 4],   # near bottom-left  -> near (0,0)m
    [590, 350, 690, 400, 9],   # near center
    [1150, 10, 1250,  60, 3],  # near top-right    -> near (30,20)m
])

real_coords = transformer.get_real_world_coords(fake_boxes)
print("Pixel centroid -> Real-world (X, Y in metres):")
for tid, (rx, ry) in real_coords.items():
    print(f"  Vehicle {tid}: ({rx:.2f}m, {ry:.2f}m)")

assert len(real_coords) == 3
print("PASS: CoordinateTransformer maps pixels to real-world metres")